# Question 1: [IPO] Withdrawn IPOs by Company Type

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import re
import requests
import io

**load data**

In [2]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

df = pd.read_html(io.StringIO(resp.text))[0]
df.head(3)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-28,Encore Medical Inc.,EMI,The Oak Ridge Financial Services Group/Network...,3.0,$5.00,$5.00,$15.00,Wednesday,S/O
1,2026-09-25,Elevation Acquisition Group,ELEVU,Maxim Group,10.0,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.0,$15.00,$18.00,$825.00,Withdrawn,S/O


In [3]:
df['Expected To Trade'].value_counts()

Expected To Trade
Priced       267
TBA          178
Withdrawn     35
Week of        7
Wednesday      5
Postponed      3
Thursday       2
Monday         1
Tuesday        1
Friday         1
Name: count, dtype: int64

In [4]:
# Фильтр: только Withdrawn, до 11.09.2026
df['File Date'] = pd.to_datetime(df['File Date'])
wd = df[(df['Expected To Trade'] == 'Withdrawn') &
       (df['File Date'] < '2026-09-11')].copy()

print(wd.shape)

(31, 10)


**company classification**

In [5]:
# --- Шаг 2: классификация (порядок правил важен — первое совпадение!) ---
def classify(name):
    clean = re.sub(r'\(.*?\)', '', str(name))
    if 'Technologies' in clean:
        return 'Technologies'
    if 'Acquisition Corp' in clean or 'Acquisition Corporation' in clean or 'Corp' in clean:
        return 'Acquisition Corp'
    if 'Inc' in clean or 'Incorporated' in clean:
        return 'Inc.'
    if 'Group' in clean:
        return 'Group'
    if 'Ltd' in clean or 'Limited' in clean:
        return 'Limited'
    if 'Holdings' in clean or 'Holding' in clean:
        return 'Holdings'
    return 'Other'

In [6]:
wd['Company Type'] = wd['Company'].apply(classify)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
376,2026-03-06,Invea Therapeutics (Withdrawn),INAI,BofA Securities/ Citigroup/ Truist Securities,0.00,NaN,NaN,$75.00,Withdrawn,S/O,Other
278,2026-05-06,Xinxu Copper Industry Technology Ltd.,XXC,Craft Capital/ R.F. Lafferty & Co.,0.00,NaN,NaN,$15.00,Withdrawn,S/O,Limited
272,2026-05-07,AiXin Life International,AIXN,Network 1 Financial Securities,2.38,$4.00,$4.00,$9.50,Withdrawn,S/O,Other
248,2026-05-20,Vistek Limited (Withdrawn),VTEK,Bancroft Capital,6.25,$4.00,$6.00,$31.00,Withdrawn,S/O,Limited
65,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O,Acquisition Corp


**price parsing**

In [7]:
def parse_price(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('$', '').replace(',', '').strip()
    if s in ('-', ''):
        return np.nan
    m = re.search(r'\d+\.?\d*', s)
    return float(m.group()) if m else np.nan

In [8]:
wd['Avg_price'] = wd[['Price Low', 'Price High']].map(parse_price).mean(axis=1)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg_price
56,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Other,NaN
58,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O,Technologies,4.5
264,2026-05-13,Ultratrex (Withdrawn),UTX,Craft Capital Management,1.25,$4.00,$6.00,$6.25,Withdrawn,S/O,Other,5.0
230,2026-06-01,Envoy Technologies (Withdrawn – Direct Listing),EVOY,Roth Capital Partners,7.80,$8.00,$8.00,$62.40,Withdrawn,S/O,Technologies,8.0
169,2026-07-08,INFINT Acquisition Corp. 2 (Withdrawn),TBA,Roth Capital Partners,10.00,$10.00,$10.00,$100.00,Withdrawn,S/O,Acquisition Corp,10.0


**numeric cols**

In [ ]:
def parse_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('$', '').replace(',', '').strip()
    return float(s) if s not in ('-', '') else np.nan

In [10]:
wd['Shares']  = wd['Shares (millions)'].apply(parse_num)
wd['Est_Vol'] = wd['Est $ Vol (millions)'].apply(parse_num)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg_price,Shares,Est_Vol
392,2026-02-24,AIGO Holding Ltd.,AIGO,Eddid Securities USA,2.00,$4.00,$6.00,$10.00,Withdrawn,S/O,Limited,5.00,2.00,10.00
49,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O,Holdings,4.50,2.00,9.00
200,2026-06-11,Club Versante Group Ltd.,CADV,Joseph Stone Capital,2.00,$4.00,$6.00,$10.00,Withdrawn,S/O,Group,5.00,2.00,10.00
264,2026-05-13,Ultratrex (Withdrawn),UTX,Craft Capital Management,1.25,$4.00,$6.00,$6.25,Withdrawn,S/O,Other,5.00,1.25,6.25
167,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,Cathay Securities,3.75,$4.00,$4.50,$15.94,Withdrawn,S/O,Holdings,4.25,3.75,15.94


**Value Calculation, aggregation**

In [11]:
wd['Shares_offered_value'] = wd['Shares'] * wd['Avg_price']
mask = wd['Shares_offered_value'].isna()
wd.loc[mask, 'Shares_offered_value'] = wd.loc[mask, 'Est_Vol']

result = (wd.groupby('Company Type')['Shares_offered_value']
          .sum().sort_values(ascending=False).round(2))
print(result)

Company Type
Acquisition Corp    499.98
Inc.                351.00
Holdings            311.66
Other               290.44
Limited             197.25
Technologies        184.90
Group                32.50
Name: Shares_offered_value, dtype: float64


# Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [12]:
url = "https://www.iposcoop.com/2025-pricings/"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
ipo = pd.read_html(io.StringIO(html))[0]
ipo.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [13]:
ipo['Offer Date'] = pd.to_datetime(ipo['Offer Date'])
ipo['Return_num'] = ipo['Return'].str.replace('%', '').astype(float)
filtered = ipo[(ipo['Offer Date'] < '2025-09-01') & (ipo['Return_num'] != 0)]
tickers = filtered['Symbol'].tolist()

In [14]:
frames = []
for t in tickers:
    d = yf.download(t, start='2025-01-01', end='2026-09-12',  # end — НЕ включительно!
                    progress=False, auto_adjust=False)
    if d is None or d.empty:
        continue                                   # делистинг / нет в Yahoo
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    d['Ticker'] = t
    frames.append(d.reset_index())

stocks_df = pd.concat(frames, ignore_index=True).sort_values(
    ['Ticker', 'Date']).reset_index(drop=True)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly d

In [15]:
g = stocks_df.groupby('Ticker')['Close']
stocks_df['growth_252d'] = stocks_df['Close'] / g.shift(252)
stocks_df['volatility']  = g.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
stocks_df[stocks_df['Date'] == '2026-09-11'].sample(5)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,growth_252d,volatility
32332,2026-09-11,10.110,10.110,15.450,8.750,12.580,6039300,PCLA,0.689162,19.597530
26353,2026-09-11,4.330,4.330,4.440,4.310,4.440,2700,MB,0.410427,17.601410
20012,2026-09-11,0.471,0.471,0.491,0.465,0.483,9200,HXHX,0.409565,0.477354
7987,2026-09-11,14.730,14.730,15.440,14.720,14.980,98900,CARL,1.083088,19.762229
20436,2026-09-11,3.280,3.280,3.374,3.210,3.300,37400,INLF,0.001005,11.457569


In [16]:
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

day = stocks_df[stocks_df['Date'] == '2026-09-11'].copy()
day['Sharpe'] = day['Sharpe'].replace([np.inf, -np.inf], np.nan)

print(day[['growth_252d', 'volatility', 'Sharpe']].describe())
print("Median Sharpe:", day['Sharpe'].median())

Price  growth_252d  volatility        Sharpe
count   130.000000  131.000000  1.280000e+02
mean      1.052117   27.571999  3.277917e+04
std       3.049217   66.868036  3.708519e+05
min       0.001005    0.000000 -4.014736e-02
25%       0.141035    2.301309  1.130346e-02
50%       0.594534    7.885748  4.722044e-02
75%       1.038821   23.968537  1.237578e-01
max      33.638270  627.698106  4.195710e+06
Median Sharpe: 0.047220436195126426


# Question 3: [IPO] 'Fixed Months Holding Strategy'

In [18]:
g = stocks_df.groupby('Ticker')['Close']
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
for m, col in zip(range(1, 13), growth_cols):
    stocks_df[col] = g.shift(-21 * m) / stocks_df['Close']

stocks_df.sample(5)

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,growth_252d,volatility,...,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
37716,2025-07-30,533.750000,533.750000,548.750000,501.250000,501.250000,383,TDIC,NaN,NaN,...,0.119438,0.075878,0.060187,0.041452,0.045667,0.031382,0.057143,0.020609,0.010023,0.004496
33738,2025-01-15,0.015000,0.015000,0.020000,0.010000,0.015000,59472,PPCB,NaN,NaN,...,18333.333743,4875.000109,8500.000190,7500.000168,9750.000218,3683.333416,2633.333392,1295.500085,1218.333335,522.500018
42126,2025-05-29,25.700001,25.700001,25.700001,24.145000,24.865000,4480,WFF,NaN,44.032993,...,0.184825,0.169261,0.153696,0.118677,0.095331,0.095136,0.079767,0.054669,0.052140,0.079767
27843,2026-06-30,37.160000,37.160000,37.240002,36.119999,36.689999,1990600,MIAX,NaN,93.579476,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3046,2026-03-05,8.390000,8.390000,8.740000,8.260000,8.740000,6400,ANTA,NaN,4.639602,...,0.866508,0.594756,0.476758,0.424315,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
min_dates = (stocks_df.groupby('Ticker', as_index=False)['Date'].min()
             .rename(columns={'Date': 'min_date'}))

entry = stocks_df.merge(min_dates,
                        left_on=['Ticker', 'Date'],
                        right_on=['Ticker', 'min_date'],
                        how='inner')
print(entry.shape)
entry.sample(5)

(132, 24)


,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,growth_252d,volatility,...,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m,min_date
77,2025-04-10,4.100000,4.100000,4.668000,3.200000,3.800000,4853600,MB,NaN,NaN,...,3.417073,2.492683,2.301220,1.829268,1.585366,1.719512,1.273171,1.378049,1.402439,2025-04-10
94,2025-05-13,7.510000,7.510000,9.230000,6.710000,8.000000,1861600,OMSE,NaN,NaN,...,0.741678,0.488682,0.680426,0.613848,0.641811,0.588549,0.589880,0.592676,0.591212,2025-05-13
112,2025-08-28,3.380000,3.380000,4.250000,3.220000,4.000000,5935800,TGHL,NaN,NaN,...,0.108580,0.195266,0.119231,0.082840,0.108284,0.399408,0.166568,0.334320,0.225148,2025-08-28
31,2025-07-23,52.799999,52.799999,63.240002,45.599998,48.000000,328417,CRE,NaN,NaN,...,0.094773,0.072500,0.067500,0.040909,0.048295,0.041004,0.042045,0.060985,0.048106,2025-07-23
105,2025-01-28,18.186831,19.750000,21.200001,19.100000,21.049999,12165200,SFD,NaN,NaN,...,1.180759,1.191392,1.231899,1.289114,1.184304,1.144304,1.083544,1.144810,1.201519,2025-01-28


In [22]:
stats = entry[growth_cols].describe()
print(stats.T[['count', 'mean', '50%']])

medians = stats.loc['50%']
best_col = medians.idxmax()
print(f"Optimal: {growth_cols.index(best_col) + 1} months, median = {medians.max():.4f}")

                    count       mean       50%
future_growth_1_m   132.0  95.751507  0.935351
future_growth_2_m   131.0  67.978625  0.892958
future_growth_3_m   131.0  51.247805  0.827160
future_growth_4_m   131.0  22.697288  0.730400
future_growth_5_m   131.0  54.242665  0.690909
future_growth_6_m   131.0  64.010844  0.700916
future_growth_7_m   131.0  59.265533  0.659973
future_growth_8_m   131.0  21.457209  0.603508
future_growth_9_m   131.0  20.006901  0.580294
future_growth_10_m  131.0  11.256621  0.533333
future_growth_11_m  131.0   8.899609  0.481811
future_growth_12_m  130.0   5.818939  0.491838
Optimal: 1 months, median = 0.9354


# Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [23]:
import gdown
from pathlib import Path

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
if not Path("data.parquet").exists():
    gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")

print(df.shape)
df.sample(5)

(229932, 203)


,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
3194,690.385772,704.673049,684.970607,688.567383,19331700.0,0.0,0.0,META,2025,2025-01-01,...,0.971646,1.048777,1.038701,0.908833,0.977751,1.010584,0.976966,1.084572,1.477935,2.377323
1470,3.734687,3.734687,3.584095,3.624253,784200.0,0.0,0.0,JPM,1986,1986-01-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3244,5.122306,5.252396,5.106045,5.252396,5953500.0,0.0,0.0,JPM,1993,1993-01-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5278,9.138470,9.315229,8.952872,9.023576,12922700.0,0.0,0.0,INFY,2020,2020-03-01,...,0.886496,0.799692,0.868677,0.656207,0.990768,1.021894,0.940705,0.940437,1.211768,2.336210
8490,39.380157,39.555474,39.226757,39.518948,12062800.0,0.0,0.0,JPM,2013,2013-11-01,...,0.977582,0.979472,1.034818,0.939929,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
RSI_THRESHOLD = 30

selected_df = df[(df['rsi'] < RSI_THRESHOLD) &
                 (df['Date'] >= '2000-01-01') &
                 (df['Date'] <= '2025-06-01')]

ret = selected_df['growth_future_30d'] - 1
net_income = 1000 * ret.sum()   

print(f"Trades: {len(selected_df)}")
print(f"Average 30-day return: {ret.mean():.2%}, win rate: {(ret > 0).mean():.2%}")
print(f"Net income: ${net_income:,.0f} (~{net_income / 1000:.1f}K)")

Сделок: 5206
Средняя доходность 30d: 1.26%, win rate: 55.13%
Net income: $65,806 (~65.8K)


# Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

**1. Manual (rule-based) filters.** Screen deals before entry using observable characteristics: skip SPACs/blank-check companies, prefer large offerings (>$300–500M) with reputable lead underwriters, require signs of strong demand at pricing (priced above the initial range, upsized deal), and check fundamentals in the S-1 filing (revenue growth, path to profitability). Even simple heuristics like these should cut out the long left tail of low-quality IPOs that drives the negative median return.

**2. Binary classification model.** Frame it as a supervised learning task: predict whether an IPO will have a positive return over a fixed horizon (e.g., 3–12 months) using features such as deal size, underwriter tier, industry, pricing vs. range, first-day return, and market regime. Because returns are right-skewed — a few big winners drive all the profit — the model should be optimized for precision in the top predictions, not overall accuracy: the goal is to identify the small subset of IPOs worth trading, not to classify every deal correctly.